# Post-Vote Small Shareholder Trading Regressions

In [ ]:
import stata_setup

stata_setup.config('/Applications/StataNow', 'mp')

In [ ]:
%%stata

clear all
set more off
set varabbrev off
version 18

* Paths
global PROCESSED_DATA "/Users/yichenluo/Library/CloudStorage/Dropbox/dao-governance/processed_data"
global TABLES "tables"
cap mkdir "$TABLES"

In [ ]:
%%stata

* Load wallet-level post-vote trading outcomes
import delimited using "$PROCESSED_DATA/post_vote_trading_wallet.csv", clear
tempfile trading
save `trading'

* Load proposal-level controls used in the existing regression tables
import delimited using "$PROCESSED_DATA/proposals_panel.csv", clear
keep id date gecko_id space topic multi_choices weighted ranked_choice quorum delegation have_discussion concensus professionalism
tempfile panel
save `panel'

use `trading', clear
merge m:1 id using `panel', keep(match) nogen

* Parse date and fixed effects
gen day = date(date, "YMD")
format day %td
gen month = month(day)
gen quarter = quarter(day)
gen year = year(day)
encode gecko_id, gen(token)
encode space,    gen(dao)
encode topic,    gen(proposal_type)

* Dependent variables
local dep bought sold traded

* Independent variables
capture confirm variable small_victory
if _rc {
    gen small_victory = non_whale_victory_vp
}
gen against_outcome = vote_against_outcome
gen log_vp = log(vp + 1)

local treatment small_victory against_outcome
local complexity multi_choices weighted ranked_choice quorum delegation
local discussion_char concensus professionalism
local wallet_char log_vp

replace concensus = concensus * have_discussion
replace professionalism = professionalism * have_discussion

* Label variables
label var bought          "\${\it Buy}_{i,j,t}\$"
label var sold            "\${\it Sell}_{i,j,t}\$"
label var traded          "\${\it Trade}_{i,j,t}\$"
label var small_victory   "\${\it Small\ Victory}_{j,t}\$"
label var against_outcome "\${\it Against\ Outcome}_{i,j,t}\$"
label var log_vp          "\${\it log}(VP_{i,j,t}+1)\$"
label var weighted        "\${\it Weighted}_{j,t}\$"
label var quadratic       "\${\it Quadratic Voting}_{j,t}\$"
label var ranked_choice   "\${\it Ranked Choice}_{j,t}\$"
label var quorum          "\${\it Quorum}_{j,t}\$"
label var multi_choices   "\${\it Multiple Choices}_{j,t}\$"
label var have_discussion "\${\it Discussion}_{j,t}\$"
label var delegation      "\${\it Delegation}_{j,t}\$"
label var professionalism "\${\it Professionalism}_{j,t}\$"
label var concensus       "\${\it Concensus}_{j,t}\$"

## Post-Vote Trading Relationship

In [ ]:
%%stata

******************************************************
* LINEAR PROBABILITY MODELS
******************************************************

eststo clear

foreach d of local dep {

    * Baseline relationship
    reghdfe `d' `treatment', absorb(year proposal_type) vce(cluster proposal_type)
    eststo `d'
    estadd local fe_type "Y"
    estadd local fe_time "Y"
    estadd local controls "N"

    * Add proposal-level controls
    reghdfe `d' `treatment' `complexity' have_discussion, absorb(year proposal_type) vce(cluster proposal_type)
    eststo `d'_c
    estadd local fe_type "Y"
    estadd local fe_time "Y"
    estadd local controls "Y"

    * Add wallet voting-power control
    reghdfe `d' `treatment' `complexity' have_discussion `wallet_char', absorb(year proposal_type) vce(cluster proposal_type)
    eststo `d'_w
    estadd local fe_type "Y"
    estadd local fe_time "Y"
    estadd local controls "Y"
}

* Export LaTeX table
esttab                                                           ///
    bought bought_c bought_w                                    ///
    sold sold_c sold_w                                          ///
    traded traded_c traded_w                                    ///
    using "$TABLES/reg_post_vote_trading.tex", replace           ///
    se star(* 0.10 ** 0.05 *** 0.01) label nogaps nocon nonumbers ///
    nonotes booktabs                                             ///
    b(%9.3f) se(%9.2f)                                           ///
    keep(small_victory against_outcome log_vp multi_choices weighted ranked_choice quorum delegation have_discussion) ///
    mgroups("\${\it Buy}_{i,j,t}\$"                              ///
            "\${\it Sell}_{i,j,t}\$"                             ///
            "\${\it Trade}_{i,j,t}\$",                           ///
            span                                                 ///
            pattern(1 0 0 1 0 0 1 0 0)                           ///
            prefix(\multicolumn{@span}{c}{)                      ///
            suffix(})                                            ///
            erepeat(\cmidrule(lr){@span}) )                      ///
    mtitles("Base" "Controls" "Wallet VP"                       ///
            "Base" "Controls" "Wallet VP"                       ///
            "Base" "Controls" "Wallet VP")                      ///
    posthead("\cmidrule(lr){2-4}\cmidrule(lr){5-7}\cmidrule(lr){8-10} &\multicolumn{1}{c}{(1)}&\multicolumn{1}{c}{(2)}&\multicolumn{1}{c}{(3)}&\multicolumn{1}{c}{(4)}&\multicolumn{1}{c}{(5)}&\multicolumn{1}{c}{(6)}&\multicolumn{1}{c}{(7)}&\multicolumn{1}{c}{(8)}&\multicolumn{1}{c}{(9)}\\\midrule") ///
    substitute("\_" "_")                                         ///
    stats(fe_type fe_time controls N r2_a,                       ///
        fmt(0 0 0 %9.0fc %9.3f)                                  ///
        labels("Proposal Type FE" "Year FE" "Controls" "Observations" "Adjusted R²"))